# LCEL + Runnables (Modern LangChain)

This notebook focuses on the LCEL programming model and core Runnable behaviors.

**Goals**
- Understand `Runnable` as the universal interface.
- Compose sequences and parallel steps.
- Use passthrough, branching, and lightweight transforms.
- Configure retries, fallbacks, and runtime config.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    RunnableBranch,
)

OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
model = ChatOpenRouter(model=OPENROUTER_MODEL, temperature=0.2)
parser = StrOutputParser()

## Runnable = universal interface
Any runnable supports `invoke`, `ainvoke`, `batch`, `stream`, and more.
You can also introspect schemas.

In [2]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a precise assistant."),
        ("human", "Explain: {topic}"),
    ]
)

chain = prompt | model | parser
chain

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a precise assistant.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain: {topic}'), additional_kwargs={})])
| ChatOpenRouter(output_version=None, client=<openrouter.sdk.OpenRouter object at 0x0000023FC7ED4D70>, openrouter_api_key=SecretStr('**********'), openrouter_api_base=None, app_url='https://docs.langchain.com', app_title='LangChain', model_name='google/gemini-3.5-flash', temperature=0.2, model_kwargs={}, session_id=None)
| StrOutputParser()

In [3]:
chain.invoke({"topic": "LCEL"})

'**LCEL** stands for **LangChain Expression Language**. \n\nIt is a declarative, unified way to compose and chain LangChain components together. It was designed to replace the older, more rigid "Chains" (like `LLMChain` or `RetrievalQA`) with a highly flexible, readable, and customizable syntax.\n\nHere is a comprehensive breakdown of what LCEL is, how it works, and why it is used.\n\n---\n\n### 1. The Core Concept: The Pipe Operator (`|`)\nLCEL uses the Unix-style pipe operator (`|`) to chain components. The output of one component is automatically passed as the input to the next.\n\nUnder the hood, LCEL relies on the **Runnable Protocol**, which standardizes how inputs and outputs flow between different objects.\n\n#### Basic Example:\n```python\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_core.output_parsers import StrOutputParser\nfrom langchain_openai import ChatOpenAI\n\n# 1. Define components\nprompt = ChatPromptTemplate.from_template("Tell me a short j

Schema introspection (handy for debugging inputs/outputs).

In [4]:
chain.input_schema.model_json_schema()

{'properties': {'topic': {'title': 'Topic', 'type': 'string'}},
 'required': ['topic'],
 'title': 'PromptInput',
 'type': 'object'}

## Parallel composition
Use a dict literal or `RunnableParallel` to fan out.

In [5]:
summarize = ChatPromptTemplate.from_template("Summarize: {text}") | model | parser
keywords = (
    ChatPromptTemplate.from_template("Extract 5 keywords: {text}") | model | parser
)

parallel = RunnableParallel(summary=summarize, keywords=keywords)

parallel.invoke({"text": "LangChain lets you build LLM apps with composable blocks."})

{'summary': '**LangChain enables modular LLM app development.**',
 'keywords': 'Here are 5 keywords extracted from the text:\n\n1. **LangChain**\n2. **LLM**\n3. **apps**\n4. **composable**\n5. **blocks**'}

## Passthrough, assign, and pick
Use `RunnablePassthrough` for light wiring without a full transform.

In [6]:
augment = (
    {
        "topic": RunnablePassthrough(),
        "style": RunnableLambda(lambda _: "concise"),
    }
    | ChatPromptTemplate.from_template("Explain {topic} in a {style} way.")
    | model
    | parser
)

augment.invoke("retrievers")

'In AI (specifically in Retrieval-Augmented Generation, or RAG), a **retriever** is a component that searches a database to find the most relevant information matching a user’s query. \n\nHere is how it works in three simple steps:\n\n1. **The Query:** You ask the AI a question.\n2. **The Search (Retriever):** The retriever scans a knowledge base (like a vector database) and pulls out the most relevant text snippets or documents.\n3. **The Answer:** It hands these snippets to the Large Language Model (LLM), which uses them to write an accurate, fact-based response.\n\n**Analogy:** Think of the **retriever** as a research assistant who finds the right library books, and the **LLM** as the writer who reads those books to draft the final report.'

## Branching and routing
Keep it simple when you only need a few routing paths.

In [7]:
short_prompt = ChatPromptTemplate.from_template("Give a short definition of {topic}.")
long_prompt = ChatPromptTemplate.from_template(
    "Write a 5-sentence explanation of {topic}."
)

short_chain = short_prompt | model | parser
long_chain = long_prompt | model | parser

router = RunnableBranch(
    (lambda x: x.get("mode") == "short", short_chain),
    (lambda x: x.get("mode") == "long", long_chain),
    short_chain,
)

router.invoke({"topic": "tool calling", "mode": "short"})

'**Tool calling** is an AI capability that allows a large language model (LLM) to recognize when it needs external information or actions, select an appropriate tool (such as an API, calculator, or database), and output the structured instructions (usually JSON) required to execute that tool. \n\nEssentially, it enables LLMs to connect to the real world and perform tasks beyond just generating text.'

## Reliability: retry + fallback
Use retries on the smallest possible runnable, not on the full pipeline.

In [8]:
import random


def flaky_transform(x: str) -> str:
    if random.random() < 0.7:
        raise ValueError("Transient failure")
    return x.upper()


flaky = RunnableLambda(flaky_transform).with_retry(stop_after_attempt=3)
safe = flaky.with_fallbacks([RunnableLambda(lambda x: f"fallback:{x}")])

safe.invoke("retry me")

'fallback:retry me'

## RunnableConfig essentials
Config lets you pass tags, metadata, and concurrency controls without polluting inputs.

In [ ]:
chain.invoke(
    {"topic": "streaming"},
    config={"tags": ["demo"], "metadata": {"notebook": "lcel"}},
)